# Credit Scoring — Fase 7: Análisis de Valor de Negocio

Convertimos el modelo de una solución técnica a una herramienta de negocio cuantificable.  

**Matriz de costos:**
| Decisión \ Realidad | No Default | Default |
|---|---|---|
| **Aprobar** | $0 (operación correcta) | -$10,000 (pérdida por impago) |
| **Rechazar** | -$500 (ingreso perdido) | +$10,000 (ahorro evitado) |

**Objetivos:**
- Encontrar el umbral de decisión que maximiza la ganancia esperada
- Cuantificar el ROI del modelo vs. la política de aprobar todo
- Construir la tabla de política crediticia por banda de riesgo

In [ ]:
import sys
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.metrics import roc_curve, confusion_matrix

warnings.filterwarnings('ignore')
sys.path.append('..')
from src.models.scorecard import Scorecard

DATA_PROC = Path('../data/processed')
MODELS    = Path('../models/saved')
FIGURES   = Path('../reports/figures')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Matriz de costos (en USD)
C_FP = -10_000   # Falso Positivo: aprobar cliente que hace default
C_FN =    -500   # Falso Negativo: rechazar cliente bueno
C_TP = +10_000   # Verdadero Positivo: rechazar cliente malo (ahorro)
C_TN =       0   # Verdadero Negativo: aprobar cliente bueno

print('Matriz de costos configurada (USD):')
print(f'  Aprobar + No Default (TN): ${C_TN:>8,.0f}')
print(f'  Aprobar + Default    (FP): ${C_FP:>8,.0f}')
print(f'  Rechazar + Default   (TP): ${C_TP:>8,.0f}')
print(f'  Rechazar + No Default(FN): ${C_FN:>8,.0f}')

## 1. Carga del modelo campeón y datos de test

In [ ]:
experiments  = json.loads((MODELS / 'experiments.json').read_text())
champion     = experiments['advanced']['champion']
file_map     = {'XGBoost': 'champion_xgb.joblib',
                'LightGBM': 'champion_lgb.joblib',
                'CatBoost': 'champion_catboost.joblib'}
model        = joblib.load(MODELS / file_map[champion])
scorecard    = Scorecard.load(MODELS / 'scorecard.joblib')

test_raw  = pd.read_csv(DATA_PROC / 'test_raw.csv')
X_test    = test_raw.drop('target', axis=1)
y_test    = test_raw['target']

y_prob    = model.predict_proba(X_test)[:, 1]
n_test    = len(y_test)

print(f'Modelo: {champion} | Test size: {n_test:,} | Default rate: {y_test.mean():.2%}')

## 2. Curva de ganancia esperada vs umbral

Para cada umbral de decisión calculamos la ganancia total si aplicáramos ese umbral al set de test.  
Identificamos el umbral que **maximiza el beneficio económico**.

In [ ]:
thresholds = np.linspace(0.01, 0.99, 500)
gains      = []

for thresh in thresholds:
    y_pred = (y_prob >= thresh).astype(int)  # 1 = rechazar
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    # tp aquí es 'rechazado y era default' → ahorro
    # fp aquí es 'rechazado y era bueno'   → ingreso perdido
    # fn aquí es 'aprobado y era default'  → pérdida
    # tn aquí es 'aprobado y era bueno'    → $0
    gain = tp * C_TP + fp * C_FN + fn * C_FP + tn * C_TN
    approval_rate = (tn + fn) / n_test
    gains.append({'threshold': thresh, 'gain': gain,
                  'approval_rate': approval_rate,
                  'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn})

gain_df        = pd.DataFrame(gains)
best_row       = gain_df.loc[gain_df['gain'].idxmax()]
optimal_thresh = best_row['threshold']
max_gain       = best_row['gain']

print(f'Umbral óptimo (máx ganancia): {optimal_thresh:.3f}')
print(f'Ganancia máxima en test:      ${max_gain:>12,.0f}')
print(f'Tasa de aprobación:           {best_row["approval_rate"]:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ganancia vs umbral
axes[0].plot(gain_df['threshold'], gain_df['gain'] / 1e6, color='steelblue', linewidth=2)
axes[0].axvline(optimal_thresh, color='tomato', linestyle='--', linewidth=1.5,
                label=f'Umbral óptimo = {optimal_thresh:.3f}')
axes[0].axhline(0, color='gray', linestyle='-', linewidth=0.8)
axes[0].set_xlabel('Umbral de decisión')
axes[0].set_ylabel('Ganancia esperada (M USD)')
axes[0].set_title('Curva ganancia vs umbral')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))

# Tasa de aprobación vs umbral
axes[1].plot(gain_df['threshold'], gain_df['approval_rate'] * 100,
             color='steelblue', linewidth=2)
axes[1].axvline(optimal_thresh, color='tomato', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Umbral de decisión')
axes[1].set_ylabel('Tasa de aprobación (%)')
axes[1].set_title('Tasa de aprobación vs umbral')

plt.tight_layout()
plt.savefig(FIGURES / '24_gain_curve.png', bbox_inches='tight')
plt.show()

## 3. ROI vs política de aprobar todo

La política naive de aprobar todo no tiene costos de rechazo pero absorbe todas las pérdidas por default.  
Comparamos cuánto valor agrega el modelo.

In [ ]:
# Política naive: aprobar todo (threshold = 0)
n_defaults = y_test.sum()
n_good     = (y_test == 0).sum()
gain_naive = n_defaults * C_FP + n_good * C_TN

# Política óptima del modelo
gain_model = max_gain

# ROI
value_added = gain_model - gain_naive
roi_pct     = (value_added / abs(gain_naive)) * 100

print('══ Comparativa de políticas ══')
print(f'  Aprobar todo (naive):          ${gain_naive:>12,.0f}')
print(f'  Modelo (umbral óptimo):        ${gain_model:>12,.0f}')
print(f'  Valor agregado por el modelo:  ${value_added:>12,.0f}')
print(f'  ROI sobre pérdida naive:       {roi_pct:>11.1f}%')

# Extrapolación a escala real (150k clientes)
scale_factor = 150_000 / n_test
print(f'\n  Extrapolación a 150k clientes:')
print(f'    Valor agregado anualizado: ${value_added * scale_factor:>12,.0f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

policies  = ['Aprobar todo\n(naive)', 'Modelo\n(umbral óptimo)']
gains_bar = [gain_naive / 1e6, gain_model / 1e6]
colors    = ['lightcoral' if g < 0 else 'steelblue' for g in gains_bar]

bars = ax.bar(policies, gains_bar, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, gains_bar):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + (0.05 if val >= 0 else -0.15),
            f'${val:.2f}M', ha='center', fontsize=11, fontweight='bold')

ax.axhline(0, color='gray', linewidth=0.8)
ax.set_ylabel('Ganancia esperada (M USD)')
ax.set_title(f'Valor agregado del modelo: ${value_added/1e6:.2f}M  ({roi_pct:.0f}% ROI)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
plt.tight_layout()
plt.savefig(FIGURES / '25_roi_comparison.png', bbox_inches='tight')
plt.show()

## 4. KPIs de negocio por banda de riesgo

Analizamos la rentabilidad de cada banda usando los scores de la scorecard.

In [ ]:
scores_df = scorecard.score_batch(X_test)
scores_df['target']     = y_test.values
scores_df['prob_model'] = y_prob

band_kpis = scores_df.groupby('risk_band').agg(
    n=('score', 'count'),
    pct_total=('score', lambda x: len(x) / n_test),
    default_rate=('target', 'mean'),
    score_mean=('score', 'mean'),
    prob_mean=('probability_default', 'mean'),
).round(4)

# Expected Loss por banda (pérdida esperada si aprobamos a todos en esa banda)
band_kpis['expected_loss_per_client'] = band_kpis['default_rate'] * abs(C_FP)
band_kpis['expected_gain_per_client'] = (1 - band_kpis['default_rate']) * C_TN + \
                                         band_kpis['default_rate'] * C_FP

print('KPIs por banda de riesgo:')
display(band_kpis[['n', 'pct_total', 'default_rate', 'score_mean',
                    'expected_loss_per_client', 'expected_gain_per_client']].round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
band_order = ['Low', 'Medium', 'High']
band_colors = {'Low': 'steelblue', 'Medium': 'orange', 'High': 'tomato'}
kpis_ordered = band_kpis.reindex(band_order)

# Volumen por banda
kpis_ordered['pct_total'].mul(100).plot(
    kind='bar', ax=axes[0],
    color=[band_colors[b] for b in band_order], edgecolor='white'
)
axes[0].set_title('% de clientes por banda')
axes[0].set_ylabel('%'); axes[0].set_xticklabels(band_order, rotation=0)

# Tasa de default por banda
kpis_ordered['default_rate'].mul(100).plot(
    kind='bar', ax=axes[1],
    color=[band_colors[b] for b in band_order], edgecolor='white'
)
axes[1].set_title('Tasa de default por banda')
axes[1].set_ylabel('%'); axes[1].set_xticklabels(band_order, rotation=0)

# Expected loss por cliente por banda
kpis_ordered['expected_loss_per_client'].plot(
    kind='bar', ax=axes[2],
    color=[band_colors[b] for b in band_order], edgecolor='white'
)
axes[2].set_title('Expected loss por cliente (USD)')
axes[2].set_ylabel('USD'); axes[2].set_xticklabels(band_order, rotation=0)
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('KPIs de negocio por banda de riesgo', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / '26_band_kpis.png', bbox_inches='tight')
plt.show()

## 5. Tabla de política crediticia

Documento final que resume la recomendación de política por banda. Este es el entregable para el equipo de riesgo.

In [ ]:
policy_table = pd.DataFrame([
    {
        'Banda':             'High',
        'Score':             '< 550',
        'Decisión':          'RECHAZAR',
        'Default rate est.': f"{kpis_ordered.loc['High', 'default_rate']:.1%}",
        'Expected loss':     f"${kpis_ordered.loc['High', 'expected_loss_per_client']:,.0f}",
        '% cartera':         f"{kpis_ordered.loc['High', 'pct_total']:.1%}",
        'Acción':            'Rechazo automático. Ofrecer producto alternativo.',
    },
    {
        'Banda':             'Medium',
        'Score':             '550 – 650',
        'Decisión':          'REVISIÓN MANUAL',
        'Default rate est.': f"{kpis_ordered.loc['Medium', 'default_rate']:.1%}",
        'Expected loss':     f"${kpis_ordered.loc['Medium', 'expected_loss_per_client']:,.0f}",
        '% cartera':         f"{kpis_ordered.loc['Medium', 'pct_total']:.1%}",
        'Acción':            'Analista revisa documentación. Posible garantía adicional.',
    },
    {
        'Banda':             'Low',
        'Score':             '> 650',
        'Decisión':          'APROBAR',
        'Default rate est.': f"{kpis_ordered.loc['Low', 'default_rate']:.1%}",
        'Expected loss':     f"${kpis_ordered.loc['Low', 'expected_loss_per_client']:,.0f}",
        '% cartera':         f"{kpis_ordered.loc['Low', 'pct_total']:.1%}",
        'Acción':            'Aprobación automática. Ofrecer mejores condiciones.',
    },
])

display(policy_table.set_index('Banda'))
policy_table.to_csv(MODELS / 'credit_policy_table.csv', index=False)
print('\nTabla guardada en models/saved/credit_policy_table.csv')

## 6. Curva de ganancia acumulada (Cumulative Gains)

¿Qué fracción de los defaults capturamos si revisamos solo el X% de los clientes más riesgosos? Cuantifica la eficiencia del modelo para priorizar revisiones.

In [ ]:
# Ordenar por probabilidad de default descendente
sorted_idx    = np.argsort(y_prob)[::-1]
sorted_target = y_test.values[sorted_idx]

total_defaults     = y_test.sum()
cum_defaults       = np.cumsum(sorted_target)
pct_population     = np.arange(1, n_test + 1) / n_test
cum_default_rate   = cum_defaults / total_defaults

# Random baseline
random_baseline = pct_population

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(pct_population * 100, cum_default_rate * 100,
        color='steelblue', linewidth=2, label='Modelo')
ax.plot([0, 100], [0, 100], 'k--', linewidth=0.8, label='Aleatorio')
ax.plot([0, total_defaults/n_test*100, 100],
        [0, 100, 100], 'g--', linewidth=0.8, label='Modelo perfecto')

# Marcar el punto 20% de población
idx_20 = np.searchsorted(pct_population, 0.20)
cap_20 = cum_default_rate[idx_20] * 100
ax.annotate(f'{cap_20:.0f}% de defaults\ncon top 20%',
            xy=(20, cap_20), xytext=(30, cap_20 - 15),
            arrowprops=dict(arrowstyle='->', color='tomato'),
            color='tomato', fontsize=9)
ax.scatter([20], [cap_20], color='tomato', zorder=5)

ax.set_xlabel('% de clientes (ordenados por riesgo desc.)')
ax.set_ylabel('% de defaults capturados')
ax.set_title('Curva de Ganancia Acumulada')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / '27_cumulative_gains.png', bbox_inches='tight')
plt.show()

print(f'Con el top 20% de clientes más riesgosos, el modelo captura el {cap_20:.0f}% de todos los defaults.')

## Resumen Fase 7

**Hallazgos clave:**
- El umbral óptimo de negocio no es 0.5 — maximizar AUC ≠ maximizar rentabilidad
- El modelo agrega valor económico significativo vs. la política naive de aprobar todo
- La banda de riesgo `High` concentra el expected loss — rechazarla protege la cartera
- Con el top 20% de clientes más riesgosos se captura la mayoría de los defaults

**Outputs:**
- `reports/figures/24–27` — curvas de negocio
- `models/saved/credit_policy_table.csv` — tabla de política crediticia

**Próximo paso:** `api/` — FastAPI para inferencia en producción